In [4]:
import numpy as np
import xarray as xr


In [14]:
eke = xr.open_dataset('../../../Med_Global_EKE/processed_data/MED/ALL_SAT/NEKE_ALL_SAT_MED_C3S_vDT2021_19930101_20230607.nc')
mask_med = np.load('../../../Med_Global_EKE/results/masks/2mask_MED_MED.npy', allow_pickle=True)
eke = eke.where(mask_med)

In [25]:
wind = xr.open_dataset('../../../Med_Global_EKE/data/era5_wind.nc')
wind = wind.drop_vars(["expver", "number"], errors="ignore").rename({"valid_time": "time", "latitude": "lat", "longitude": "lon"})
wind = wind.sel(time=slice(None, "2023-06-07"))
wind = wind.sortby("lat")
wind = wind.rename({"avg_iews": "surfu", "avg_inss": "surfv"})

In [46]:
R = 6371000

lat = wind.lat.values
lon = wind.lon.values

lat_rad = np.deg2rad(lat)
lon_rad = np.deg2rad(lon)

dlat = np.gradient(lat_rad)
dlon = np.gradient(lon_rad)

# dx
dx = R * np.cos(lat_rad)[:, None] * dlon
dx = xr.DataArray(dx, coords={"lat": wind.lat, "lon": wind.lon}, dims=("lat", "lon"))

# dy
dy = xr.DataArray(R * dlat, coords={"lat": wind.lat}, dims=("lat"))
dy = dy.broadcast_like(wind.surfu)

# dérivées
d_tauy_dlon = wind.surfv.differentiate("lon")
d_taux_dlat = wind.surfu.differentiate("lat")

# curl
curl = (d_tauy_dlon / dx) - (d_taux_dlat / dy)

wind["curl"] = curl
wind["curl"].attrs["units"] = "N m^-3"

In [ ]:
def subset_region(ds, lonmin, lonmax, latmin, latmax):
    if ds.lat[0] < ds.lat[-1]:
        lat_slice = slice(latmin, latmax)
    else:
        lat_slice = slice(latmax, latmin)

    return ds.sel(lon=slice(lonmin, lonmax), lat=lat_slice)

In [52]:
wind_alb = subset_region(wind, lonmin=-5.4, lonmax=-1, latmin=35.2, latmax=36.7)
eke_alb = subset_region(eke, lonmin=-5.4, lonmax=-1, latmin=35.2, latmax=36.7)

wind_iera = subset_region(wind, lonmin=24, lonmax=31, latmin=31, latmax=35.5)
eke_iera = subset_region(eke, lonmin=24, lonmax=31, latmin=31, latmax=35.5)


In [ ]:
np

In [ ]:
box = wind.sel(lon=slice(...), lat=slice(...))
curl_index = box.curl.mean(("lat", "lon"))

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 11115, lat: 69, lon: 173)
Coordinates:
  * time     (time) datetime64[ns] 89kB 1993-01-01 1993-01-02 ... 2023-06-07
  * lat      (lat) float64 552B 30.0 30.25 30.5 30.75 ... 46.25 46.5 46.75 47.0
  * lon      (lon) float64 1kB -6.0 -5.75 -5.5 -5.25 ... 36.25 36.5 36.75 37.0
Data variables:
    surfu    (time, lat, lon) float32 531MB 0.0004044 -0.0005722 ... -0.004248
    surfv    (time, lat, lon) float32 531MB 0.02745 0.02403 ... -0.04316 -0.0412
    curl     (time, lat, lon) float64 1GB -2.851e-06 -9.278e-07 ... -4.461e-08
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-03-20T09:29 GRIB to CDM+CF via cfgrib-0.9.1...